# 01 Exploratory Data Analysis for BUT PPG

This notebook audits the dataset before modeling. It verifies label and
subject distributions, explores motion and reference heart rate, and
visually inspects both legacy and RGB waveform layouts.

**Rule:** EDA may inspect labels across the complete dataset, but model
selection and threshold decisions must use only train and validation data.

In [ ]:
from pathlib import Path
import os
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "apps").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the pure-pulsesure repository")


REPO_ROOT = find_repository_root(Path.cwd().resolve())
AI_SERVICE_ROOT = REPO_ROOT / "apps" / "ai-service"
sys.path.insert(0, str(AI_SERVICE_ROOT))

RAW_DIR = Path(
    os.environ.get("PURE_BUT_PPG_DIR", REPO_ROOT / "data" / "raw" / "but-ppg-2.0.0")
).expanduser().resolve()
PROCESSED_DIR = Path(
    os.environ.get("PURE_PROCESSED_DIR", REPO_ROOT / "data" / "processed" / "but-ppg-v2")
).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PURE_MODEL_DIR", REPO_ROOT / "models")).expanduser().resolve()

print("Repository:", REPO_ROOT)
print("Raw dataset:", RAW_DIR)
print("Processed dataset:", PROCESSED_DIR)
print("Model directory:", MODEL_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from app.services.preprocessing import load_but_ppg_record, preprocess_ppg

sns.set_theme(style="whitegrid")
annotations = pd.read_csv(RAW_DIR / "quality-hr-ann.csv", encoding="utf-8-sig")
subject_info = pd.read_csv(RAW_DIR / "subject-info.csv", encoding="utf-8-sig")
annotations["ID"] = annotations["ID"].astype(str).str.zfill(6)
subject_info["ID"] = subject_info["ID"].astype(str).str.zfill(6)
annotations["subject_id"] = annotations["ID"].str[:3]
dataset = annotations.merge(subject_info, on="ID", how="left")
dataset.head()

In [ ]:
summary = pd.Series({
    "records": len(dataset),
    "subjects": dataset["subject_id"].nunique(),
    "good_quality": int((dataset["Quality"] == 1).sum()),
    "poor_quality": int((dataset["Quality"] == 0).sum()),
    "missing_reference_hr": int(dataset["HR"].isna().sum()),
    "duplicate_record_ids": int(dataset["ID"].duplicated().sum()),
})
display(summary.to_frame("value"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=dataset, x="Quality", ax=axes[0], palette="Set2", hue="Quality", legend=False)
axes[0].set_title("Record-level quality labels")
axes[0].set_xticks([0, 1], ["Poor", "Good"])

records_per_subject = dataset.groupby("subject_id").size()
sns.histplot(records_per_subject, bins=15, ax=axes[1])
axes[1].set_title("Records per subject")
axes[1].set_xlabel("Record count")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=dataset, x="HR", hue="Quality", bins=25, element="step", ax=axes[0])
axes[0].set_title("Reference HR by quality label")

motion_quality = pd.crosstab(dataset["Motion"].fillna("unknown").astype(str), dataset["Quality"], normalize="index")
motion_quality.columns = ["Poor", "Good"]
motion_quality.sort_values("Good").plot(kind="bar", stacked=True, ax=axes[1], color=["#E76F51", "#2A9D8F"])
axes[1].set_title("Quality fraction by motion annotation")
axes[1].set_ylabel("Fraction")
axes[1].legend(loc="lower right")
plt.tight_layout()

In [ ]:
legacy = dataset[dataset["ID"].astype(int) < 112000]
modern = dataset[dataset["ID"].astype(int) >= 112000]
preview_ids = [
    legacy.loc[legacy["Quality"] == 1, "ID"].iloc[0],
    legacy.loc[legacy["Quality"] == 0, "ID"].iloc[0],
    modern.loc[modern["Quality"] == 1, "ID"].iloc[0],
    modern.loc[modern["Quality"] == 0, "ID"].iloc[0],
]

fig, axes = plt.subplots(len(preview_ids), 2, figsize=(14, 10), sharex=True)
for row_index, record_id in enumerate(preview_ids):
    record = load_but_ppg_record(RAW_DIR / record_id)
    processed = preprocess_ppg(record.waveform)
    time = np.arange(record.waveform.size) / record.sampling_rate_hz
    label = int(dataset.loc[dataset["ID"] == record_id, "Quality"].iloc[0])
    axes[row_index, 0].plot(time, record.waveform, color="#142638")
    axes[row_index, 0].set_title(f"{record_id} raw | label={label} | {record.selected_channel}")
    axes[row_index, 1].plot(time, processed, color="#147D78")
    axes[row_index, 1].set_title(f"{record_id} preprocessed")
axes[-1, 0].set_xlabel("Seconds")
axes[-1, 1].set_xlabel("Seconds")
plt.tight_layout()

## Interpretation checklist

Record observed imbalance, suspicious label or HR values, motion-related
shifts, and visual differences between good and poor signals here. Do not
convert these observations into test-set-driven modeling decisions.